<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔧 用 Middleware 管理消息

**一句话定位**：agent 跑得越久，`messages[]` 越长——middleware 就是在它变成问题之前，把消息历史拦下来改一改的钩子。

</div>

### 为什么需要 middleware？

agent 本质是一个循环：用户发消息 → LLM 响应（可能调用工具）→ 工具结果回来 → LLM 再响应……每一轮都在往 `messages[]` 里追加。轮次多了会出三个问题：

1. **上下文溢出** —— 再大的上下文窗口（128k+）长会话也会撑爆
2. **成本** —— prompt 里每个 token 每次调用都要计费，第 51 轮要为前 50 轮历史付费
3. **质量下降** —— "lost in the middle" 现象：关键信息埋得越深，LLM 推理越差，上下文越短越聚焦，效果越好

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**Middleware** 能在 agent 循环的特定节点上拦截并改写消息历史——在这些问题传到 LLM 之前就处理掉。

</div>

### node 风格 middleware 是什么？

LangChain/LangGraph agent 本质是一张由 **node**（LLM 调用节点、工具执行节点等）和 edge 组成的图。node 风格 middleware 就是挂在这些节点切换处的钩子：

![Agent Loop with Middleware Hooks](resources/agent_diagram_small_arrows_FIXED.svg)

| Hook | 触发时机 | 常见用途 |
|---|---|---|
| `@before_agent` | LLM 被调用之前 | 裁剪消息、总结历史、注入系统上下文 |
| `@after_agent` | LLM 响应之后 | 记录响应、校验输出、修改 AI 消息 |
| `@before_tools` | 工具执行之前 | 校验工具调用、加授权检查 |
| `@after_tools` | 工具返回之后 | 改写工具结果、缓存响应 |

middleware 接收当前的 `AgentState`（包含 `messages[]` 和自定义 state 字段），返回修改内容；返回 `None` 则状态原样通过。

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：查了一下 LangChain 官方文档（docs.langchain.com / reference.langchain.com），目前实际提供的 node 风格 hook 是 **`before_agent`、`before_model`、`after_model`、`after_agent`** 这四个——`before_agent`/`after_agent` 只在整次 agent 运行的开头/结尾各跑一次（适合做初始化、加载文件），而 `before_model`/`after_model` 才是**每次** LLM 调用前后都会跑（适合裁剪消息、做总结、人工审核）。并**没有** `before_tools`/`after_tools` 这两个钩子——工具层面目前只能用 `wrap_tool_call` 做拦截式包装，官方已有 issue 在讨论要不要补上对称的 observational hook，但截至目前还没加。下面几个 cell 里用到的 `@before_agent` 本身是真实存在的 API，用法没问题，只是这张总览表把 tool 那两格的名字写错了。

</div>

### 本节两种模式

**模式 1**：`SummarizationMiddleware` —— 内置的自动总结中间件
**模式 2**：自定义 `@before_agent` —— 手写裁剪/删除消息的逻辑

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 总结消息（Summarize messages）

### 🧠 `SummarizationMiddleware` 是什么？

agent 跑得越久 `messages[]` 越长，直接截断旧消息会丢上下文。`SummarizationMiddleware` 是折中方案：

1. ⏱️ 监控消息数（或 token 数）
2. 📝 超过阈值时，让 LLM 把旧消息**总结**成一条压缩消息
3. 🔄 用总结替换掉那些旧消息
4. ✅ 保留最近的消息不动

于是

```
[msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]  ← 一直在长
```

变成

```
[summary_of_1_through_5, msg6, msg7, msg8]  ← 大小有界
```

agent 依然"记得"早期发生的事（通过总结），但实际 token 数可控。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：这本质上就是**对自己的对话历史做自动 RAG**。

</div>

**下面代码里的关键参数：**
- `model="gpt-4o-mini"` —— 一个便宜快速的模型，专门用来做总结（不是 agent 主模型）
- `trigger=("tokens", 100)` —— 消息超过 100 token 时触发总结
- `keep=("messages", 1)` —— 永远保留最近 1 条消息不动

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is asking about various details regarding Lunapolis, a fictional city on the moon, including its capital, weather, population of cheese miners, and potential union activities.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear skies, with temperatures ranging from a high of 120C to a low of -100C.\n- Lunapolis is home to 100,000 cheese miners.\n- There is speculation that the cheese miners' union may strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='6b927c78-d20b-4eec-b866-60f9cbf5d0db'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='e6bc9308-d78e-419e-9e62

In [ ]:
print(response["messages"][0].content)

## 裁剪 / 删除消息（Trim/delete messages）

In [4]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [6]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(
                content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"
            ),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on."),
            ToolMessage(
                content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"
            ),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?"),
        ]
    },
    {"configurable": {"thread_id": "2"}},
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5a559ee9-a4c7-4203-bca6-b782fef85e40'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='74551317-3878-48e4-96ec-b401b5dc3f2e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='6a344bfd-8d7a-4f4f-851d-4ea4d0ec576a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0e3579d1-ab92-4f4c-a677-d8c4c8b0b654', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='1697ddf0-a447-4a92-b3d6-2cfa1962f2ae'),
              AIMessage(content='I can’t read the device’s temperature from here. If you want t

In [7]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If you want to check manually, try these:

- Feel for heat: is the back/bottom or charger area hot to the touch? If yes, it may be overheating.
- If it’s hot: power it off, unplug it, and let it cool on a hard surface with good ventilation for 15–30 minutes.
- Check ventilation: make sure vents aren’t blocked by dust, fabric, or furniture. 
- Inspect for dust and debris and consider cleaning the vents with compressed air (while powered off and unplugged).
- After it’s cooled, try powering it on again. If it still won’t start or repeatedly overheats, try a different charger/outlet.
- If you notice swelling, burning smell, or it gets extremely hot quickly, stop using it and contact support.

If you can share the device type (brand/model) and OS, I can give more specific steps to check temperatures and diagnose boot issues.
